# SmolVLM2-2.2B-Instruct – POPE Inference

Notebook thực hiện quá trình chuẩn bị dữ liệu POPE, khởi tạo mô hình SmolVLM2-2.2B-Instruct và chạy benchmark trên tập test.

**Đầu ra:** `pope_smollvm2_checkpoint.csv` và `pope_smollvm2_checkpoint.json`.

> Chạy các cell theo thứ tự từ trên xuống. Cell cài đặt thư viện chỉ cần chạy khi môi trường chưa có các thư viện tương ứng; sau khi cài đặt, khởi động lại session nếu Kaggle yêu cầu.

## Bước 2. Kiểm tra thông tin môi trường thực thi

Kiểm tra phiên bản Python, hệ điều hành, PyTorch và thông tin GPU được Kaggle cung cấp. Qua đó xác nhận môi trường đã đáp ứng yêu cầu để tải và chạy mô hình SmolVLM2-2.2B-Instruct.

In [ ]:

import sys
import torch
import platform

print("========== THÔNG TIN MÔI TRƯỜNG ==========")

print("Python:", sys.version)
print("Hệ điều hành:", platform.platform())
print("PyTorch:", torch.__version__)

# Kiểm tra GPU
if torch.cuda.is_available():
    print("\nGPU khả dụng: Có")
    print("Tên GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

    total_memory = torch.cuda.get_device_properties(0).total_memory
    print("Dung lượng GPU:",
          round(total_memory / (1024 ** 3), 2), "GB")
else:
    print("\nGPU khả dụng: Không")
    print("Vui lòng bật GPU trong Kaggle Settings.")

## Bước 3. Kiểm tra thư viện và thiết bị thực thi

Kiểm tra thư viện `num2words` và khả năng nhận diện thư viện từ Transformers. Đồng thời, xác định thiết bị thực thi là CPU hay GPU để bảo đảm môi trường đã sẵn sàng cho quá trình tải và chạy mô hình SmolVLM2-2.2B-Instruct.

In [ ]:
# Cai thu vien (chay 1 lan, sau do Restart Session roi chay tiep)
%pip install -q --upgrade "transformers>=4.49" accelerate num2words safetensors pillow scikit-learn
print("Da cai xong. Hay Restart Session truoc khi chay tiep.")

In [ ]:
import gc
import torch
import num2words

# Kiểm tra num2words trước
print("num2words:", num2words.num2words(22))

# Import Transformers sau khi num2words đã sẵn sàng
from transformers import AutoProcessor, AutoModelForImageTextToText

from transformers.utils import import_utils

print("Transformers nhận diện num2words:",
      import_utils.is_num2words_available())

# Kiểm tra thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import transformers
import tokenizers
import huggingface_hub
import datasets

print("===== KIỂM TRA MÔI TRƯỜNG =====")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Bước 4. Kiểm tra phiên bản thư viện

Kiểm tra phiên bản của các thư viện chính được sử dụng trong quá trình thực nghiệm, bao gồm PyTorch, Transformers, Tokenizers, HuggingFace Hub và Datasets. Đồng thời, xác nhận khả năng sử dụng CUDA và thông tin GPU nhằm hạn chế các lỗi không tương thích trong quá trình chạy mô hình.

In [ ]:
import gc
import torch
import num2words

from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Thiết bị:", device)

# Giải phóng bộ nhớ
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Chọn kiểu dữ liệu
dtype = torch.float16 if device == "cuda" else torch.float32

try:
    print("\nĐang tải Processor...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)

    print("Processor đã tải thành công!")

    print("\nĐang tải mô hình SmolVLM2...")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None,
        low_cpu_mem_usage=True
    )

    model.eval()

    print("\nTẢI MÔ HÌNH THÀNH CÔNG!")
    print("Tên mô hình:", MODEL_NAME)
    print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
    print("Kiểu dữ liệu:", dtype)

except Exception as e:
    print("\nLỖI:", type(e).__name__, str(e))
    raise

TẢI VÀ KHẢO SÁT DATASET POPE

Trong bước này, tiến hành tải bộ dữ liệu POPE từ Hugging Face. Đây là bộ dữ liệu được sử dụng để đánh giá hiện tượng Hallucination thông qua các câu hỏi xác định sự tồn tại của đối tượng trong hình ảnh.



In [ ]:


from datasets import load_dataset

DATASET_NAME = "lmms-lab-encoder/POPE"

print("Đang tải dataset POPE...")

dataset = load_dataset(DATASET_NAME)

print("\nTải dataset thành công!")
print(dataset)

In [ ]:


print("Các tập dữ liệu:", list(dataset.keys()))

for split in dataset.keys():
    print(f"\nTập: {split}")
    print("Số lượng mẫu:", len(dataset[split]))
    print("Các cột:", dataset[split].column_names)

# Hiển thị một mẫu dữ liệu
sample = dataset[list(dataset.keys())[0]][0]

print("\nMẪU DỮ LIỆU:")
for key, value in sample.items():
    if key == "image":
        print(key, ":", value)
    else:
        print(key, ":", value)

 PHÂN TÍCH VÀ KIỂM TRA DATASET POPE

Tiến hành khảo sát dữ liệu nhằm xác định phân bố nhãn Yes/No và các nhóm câu hỏi trong POPE. Đồng thời, kiểm tra định dạng hình ảnh để đảm bảo dữ liệu phù hợp với đầu vào của mô hình SmolVLM2.



In [ ]:

from collections import Counter
import pandas as pd

test_data = dataset["test"]

# Thống kê nhãn thực tế
answer_counts = Counter(
    str(item).strip().lower() for item in test_data["answer"]
)

# Thống kê nhóm câu hỏi
category_counts = Counter(test_data["category"])

print("========== PHÂN BỐ NHÃN ==========")

for label, count in answer_counts.items():
    print(f"{label}: {count} mẫu ({count / len(test_data) * 100:.2f}%)")

print("\n========== PHÂN BỐ CATEGORY ==========")

for category, count in category_counts.items():
    print(f"{category}: {count} mẫu ({count / len(test_data) * 100:.2f}%)")

# Tạo bảng thống kê
summary = pd.DataFrame({
    "Nhãn": list(answer_counts.keys()),
    "Số lượng": list(answer_counts.values())
})

display(summary)

In [ ]:


missing_counts = {
    column: sum(value is None for value in test_data[column])
    for column in test_data.column_names
}

print("========== KIỂM TRA DỮ LIỆU THIẾU ==========")

for column, count in missing_counts.items():
    print(f"{column}: {count} giá trị thiếu")

# Kiểm tra hình ảnh
image = test_data[0]["image"]

print("\n========== THÔNG TIN HÌNH ẢNH ==========")
print("Định dạng:", image.format)
print("Chế độ màu:", image.mode)
print("Kích thước:", image.size)

# Hiển thị hình ảnh mẫu
display(image)

CHUẨN BỊ DỮ LIỆU ĐẦU VÀO

Ở bước này, tiến hành chuẩn hóa dữ liệu POPE trước khi đưa vào mô hình SmolVLM2.



In [ ]:

from PIL import Image

test_data = dataset["test"]

benchmark_data = []

for idx, sample in enumerate(test_data):

    # Chuẩn hóa hình ảnh
    image = sample["image"].convert("RGB")

    # Chuẩn hóa câu hỏi
    question = str(sample["question"]).strip()

    # Chuẩn hóa nhãn thực tế
    answer = str(sample["answer"]).strip().lower()

    # Chỉ giữ các nhãn hợp lệ
    if answer not in ["yes", "no"]:
        continue

    benchmark_data.append({
        "id": f'{sample["category"]}_{sample["id"]}',
        "question": question,
        "image": image,
        "answer": answer,
        "category": sample["category"],
        "image_source": sample["image_source"]
    })

print("========== KẾT QUẢ TIỀN XỬ LÝ ==========")
print("Tổng số mẫu ban đầu:", len(test_data))
print("Số mẫu hợp lệ:", len(benchmark_data))
print("Số mẫu bị loại:", len(test_data) - len(benchmark_data))

In [ ]:


sample = benchmark_data[0]

print("ID:", sample["id"])
print("So id duy nhat:", len({d["id"] for d in benchmark_data}), "/", len(benchmark_data))
print("Question:", sample["question"])
print("Ground Truth:", sample["answer"])
print("Category:", sample["category"])
print("Image Source:", sample["image_source"])
print("Image Size:", sample["image"].size)

display(sample["image"])

 THIẾT LẬP QUY TRÌNH SUY LUẬN

Trong bước này, xây dựng hàm Inference để đưa hình ảnh và câu hỏi từ POPE vào mô hình SmolVLM2-2.2B-Instruct.

Mô hình được yêu cầu xác định đối tượng có xuất hiện trong hình ảnh hay không và chỉ trả lời bằng một trong hai nhãn Yes hoặc No.

Quy trình thực hiện:
1. Nhận hình ảnh và câu hỏi.
2. Chuyển dữ liệu thành định dạng đầu vào của SmolVLM2.
3. Thực hiện sinh câu trả lời.
4. Giải mã kết quả đầu ra.
5. Chuẩn hóa câu trả lời thành Yes hoặc No.

In [ ]:
# BƯỚC 7: KIỂM TRA KÍCH THƯỚC ĐẦU VÀO CỦA SMOLVLM2

import torch
import transformers

sample = benchmark_data[0]
image = sample["image"].convert("RGB")
question = sample["question"]

print("Transformers version:", transformers.__version__)
print("Image size:", image.size)
print("Image mode:", image.mode)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": f"{question}\nAnswer only Yes or No."
            }
        ]
    }
]

prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True
)

inputs = processor(
    text=prompt,
    images=image,
    return_tensors="pt"
)

print("\n========== INPUT SHAPES ==========")

for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(key, value.shape, value.dtype)

print("\n========== MODEL CONFIG ==========")
print("Patch size:", model.config.vision_config.patch_size)
print("Hidden size:", model.config.vision_config.hidden_size)

In [ ]:

# BƯỚC 8: HÀM DỰ ĐOÁN SMOLVLM2 ĐÃ ĐIỀU CHỈNH

import torch
import time

def predict_smollvm(image, question):
    image = image.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": (
                        f"{question}\n"
                        "Answer only Yes or No. "
                        "Base your answer strictly on the image."
                    )
                }
            ]
        }
    ]

    # Xử lý ảnh và văn bản trực tiếp qua chat template
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # Sinh câu trả lời
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    # Chỉ giải mã phần token mới được sinh
    input_length = inputs["input_ids"].shape[1]

    response = processor.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    return response


 CHUẨN HÓA CÂU TRẢ LỜI

Câu trả lời của SmolVLM2 có thể xuất hiện dưới nhiều dạng như Yes, Yes., No hoặc No.

Trong bước này, xây dựng hàm chuẩn hóa kết quả thành hai nhãn thống nhất:
- yes: Đối tượng xuất hiện trong hình ảnh.
- no: Đối tượng không xuất hiện trong hình ảnh.
- unknown: Câu trả lời không xác định được hai nhãn trên.

Việc chuẩn hóa giúp đảm bảo tính nhất quán khi đối chiếu dự đoán với nhãn thực tế.

In [ ]:

import re

def normalize_answer(response):
    text = response.strip().lower()

    # Tìm câu trả lời Yes hoặc No ở đầu phản hồi
    match = re.match(r"^(yes|no)\b", text)

    if match:
        return match.group(1)

    return "unknown"


# Kiểm tra hàm chuẩn hóa
test_responses = [
    "Yes",
    "Yes.",
    "No",
    "No.",
    "The answer is yes.",
    "I cannot determine."
]

for response in test_responses:
    print(f"{response:25} -> {normalize_answer(response)}")

In [ ]:
sample = benchmark_data[0]
prediction = predict_smollvm(sample["image"], sample["question"])
normalized_prediction = normalize_answer(prediction)

print("Ground Truth:", sample["answer"])
print("Raw Prediction:", prediction)
print("Normalized Prediction:", normalized_prediction)

if normalized_prediction == sample["answer"]:
    print("Ket qua: DUNG")
elif normalized_prediction == "unknown":
    print("Ket qua: KHONG XAC DINH")
else:
    print("Ket qua: SAI")

THỰC HIỆN BENCHMARK TRÊN DATASET POPE

Tiến hành đưa toàn bộ 9.000 mẫu trong tập test POPE vào mô hình SmolVLM2-2.2B-Instruct để đánh giá khả năng nhận diện đối tượng trong hình ảnh.

Với mỗi mẫu, mô hình sẽ:
- Nhận hình ảnh và câu hỏi tương ứng.
- Sinh câu trả lời Yes/No.
- Chuẩn hóa câu trả lời.
- Đối chiếu với nhãn thực tế.
- Ghi nhận thời gian suy luận.

Kết quả được lưu lại để sử dụng trong quá trình tính toán các chỉ số đánh giá.

In [ ]:
# BƯỚC 11.1: KIỂM TRA VÀ KHỞI TẠO CHECKPOINT

import os
import pandas as pd

CHECKPOINT_PATH = "/kaggle/working/pope_smollvm2_checkpoint.csv"
JSON_PATH = "/kaggle/working/pope_smollvm2_checkpoint.json"

benchmark_results = []
completed_ids = set()

if os.path.exists(CHECKPOINT_PATH) and os.path.getsize(CHECKPOINT_PATH) > 0:
    try:
        old_df = pd.read_csv(CHECKPOINT_PATH)

        if "id" in old_df.columns:
            done = old_df[old_df["status"].astype(str).isin(["success","unknown_response"])]
            benchmark_results = done.to_dict("records")
            completed_ids = set(done["id"].astype(str).tolist())

        print("Đã khôi phục:", len(benchmark_results), "kết quả")

    except pd.errors.EmptyDataError:
        print("Checkpoint rỗng, sẽ bắt đầu benchmark mới.")

else:
    print("Chưa có checkpoint hợp lệ, sẽ bắt đầu benchmark mới.")

print("Số kết quả hiện tại:", len(benchmark_results))
print("Số mẫu đã hoàn thành:", len(completed_ids))
print("Số mẫu còn lại:", len(benchmark_data) - len(completed_ids))

In [ ]:
# BƯỚC 11.3: CHẠY BENCHMARK POPE

import os
import json
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

CHECKPOINT_PATH = "/kaggle/working/pope_smollvm2_checkpoint.csv"
JSON_PATH = "/kaggle/working/pope_smollvm2_checkpoint.json"

SAVE_INTERVAL = 10

# Khởi tạo kết quả
benchmark_results = []
completed_ids = set()
# Khoi phuc tu checkpoint (khong ghi de)
if os.path.exists(CHECKPOINT_PATH) and os.path.getsize(CHECKPOINT_PATH) > 0:
    _old = pd.read_csv(CHECKPOINT_PATH)
    _done = _old[_old["status"].astype(str).isin(["success", "unknown_response"])]
    benchmark_results = _done.to_dict("records")
    completed_ids = set(_done["id"].astype(str))
    print("Da khoi phuc", len(completed_ids), "mau tu checkpoint")

# Lưu checkpoint an toàn
def save_checkpoint():
    if not benchmark_results:
        return

    df = pd.DataFrame(benchmark_results)

    temp_csv = CHECKPOINT_PATH + ".tmp"
    temp_json = JSON_PATH + ".tmp"

    df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
    os.replace(temp_csv, CHECKPOINT_PATH)

    with open(temp_json, "w", encoding="utf-8") as f:
        json.dump(benchmark_results, f, ensure_ascii=False, indent=2)

    os.replace(temp_json, JSON_PATH)

# Chạy benchmark
model.eval()

start_time = time.time()
new_count = 0
error_count = 0

try:
    for item in tqdm(benchmark_data, desc="POPE Benchmark"):

        sample_id = str(item["id"])

        if sample_id in completed_ids:
            continue

        inference_start = time.time()

        try:
            raw_response = predict_smollvm(
                item["image"],
                item["question"]
            )

            prediction = normalize_answer(raw_response)
            status = "success"

            if prediction == "unknown":
                status = "unknown_response"

        except Exception as e:
            raw_response = ""
            prediction = "unknown"
            status = f"error: {str(e)[:250]}"
            error_count += 1

        inference_time = time.time() - inference_start

        benchmark_results.append({
            "id": item["id"],
            "question": item["question"],
            "ground_truth": item["answer"],
            "raw_response": raw_response,
            "prediction": prediction,
            "category": item["category"],
            "inference_time": round(inference_time, 4),
            "status": status
        })

        if not status.startswith("error"):
            completed_ids.add(sample_id)
        new_count += 1

        # Lưu mỗi 10 mẫu
        if new_count % SAVE_INTERVAL == 0:
            save_checkpoint()

        # Hiển thị tiến độ mỗi 50 mẫu
        if new_count % 50 == 0:
            print(
                f"Đã xử lý: {len(benchmark_results)}/9000 | "
                f"Lỗi: {error_count}"
            )

except KeyboardInterrupt:
    print("\nBenchmark đã được dừng thủ công.")

finally:
    save_checkpoint()

    elapsed = time.time() - start_time

    print("\n========== BENCHMARK STATUS ==========")
    print("Đã xử lý:", len(benchmark_results), "/ 9000")
    print("Lỗi:", error_count)
    print("Thời gian:", round(elapsed / 60, 2), "phút")
    print("Checkpoint:", CHECKPOINT_PATH)
    print("Đã lưu kết quả.")

## Kiểm tra file đầu ra

Sau khi benchmark kết thúc, kiểm tra trong `/kaggle/working/`:

- `pope_smollvm2_checkpoint.csv`: kết quả dự đoán theo từng mẫu.
- `pope_smollvm2_checkpoint.json`: dữ liệu kết quả dạng JSON.

Chỉ chuyển sang Notebook Evaluate khi file CSV đã có dữ liệu và benchmark đã chạy đến mức mong muốn.